In [ ]:


import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import os

def main():
    """
    Main function to run the complete machine learning pipeline.
    """
    # 1. Load the data from the specified Kaggle input paths
    print("Loading data...")
    try:
        train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
        test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")
    except FileNotFoundError as e:
        print(f"Error: The specified file was not found. Please ensure the files are available at the correct paths: {e}")
        return

    # 2. Preprocessing and Feature Engineering
    print("Preprocessing data...")
    
    # Fill any potential NaN values in the text columns with an empty string
    train_df = train_df.fillna('')
    test_df = test_df.fillna('')

    # Combine the prompt and both responses into a single text feature for the model.
    # This allows the model to learn relationships between the three text fields.
    train_df['text'] = train_df['prompt'] + " " + train_df['response_a'] + " " + train_df['response_b']
    test_df['text'] = test_df['prompt'] + " " + test_df['response_a'] + " " + test_df['response_b']

    # Identify the target column by finding the column with a value of 1.
    # The columns are winner_model_a, winner_model_b, and winner_tie.
    # We'll convert these one-hot encoded columns into a single categorical column.
    train_df['winner'] = train_df[['winner_model_a', 'winner_model_b', 'winner_tie']].idxmax(axis=1)

    # 3. Vectorization
    print("Vectorizing text data...")
    # TfidfVectorizer converts a collection of raw documents to a matrix of TF-IDF features.
    # It assigns a weight to each word based on its frequency in a document and its rarity across all documents.
    vectorizer = TfidfVectorizer(max_features=5000)
    
    # Fit the vectorizer on the combined text from both train and test sets to ensure a consistent vocabulary.
    full_text = pd.concat([train_df['text'], test_df['text']], ignore_index=True)
    vectorizer.fit(full_text)

    X_train = vectorizer.transform(train_df['text'])
    X_test = vectorizer.transform(test_df['text'])
    y_train = train_df['winner']

    # 4. Model Training
    print("Training Logistic Regression model...")
    # Logistic Regression is a simple but effective linear model for classification.
    # It is a good baseline to start with.
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    # 5. Prediction
    print("Making predictions on the test set...")
    # Predict the probabilities for each class (winner_model_a, winner_model_b, winner_tie)
    test_predictions_proba = model.predict_proba(X_test)

    # 6. Create Submission File
    print("Creating submission file...")
    # The classes are returned in alphabetical order by default by scikit-learn.
    # We need to ensure the columns in the submission file are in the correct order.
    class_labels = model.classes_
    submission_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
    
    # Create the submission DataFrame with the correct structure
    submission_df = pd.DataFrame(test_predictions_proba, columns=class_labels)
    
    # Reorder columns to match the required submission format
    submission_df = submission_df[submission_cols]
    submission_df.insert(0, 'id', test_df['id'])
    
    # Save the submission file
    submission_df.to_csv("submission.csv", index=False)
    print("Submission file created successfully: submission.csv")

if __name__ == "__main__":
    main()
